### Passo 1: Processamento de Dados com spaCy

#### Instalação de Ambiente:

In [ ]:
%pip install pandas spacy scikit-learn matplotlib seaborn wordcloud

In [ ]:
!python -m spacy download en_core_web_sm

#### Importação de Libs

In [ ]:
import pandas as pd
import spacy
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix
from wordcloud import WordCloud 

nlp = spacy.load("en_core_web_sm")

#### Criação e Consumo dos Datasets

In [ ]:
# Utilização de Dataset Local (arquivo bruto)
df = pd.read_csv('../../dataset/raw/phishing_emails.csv', encoding='utf-8')  

# Visualização prévia dele
df.head()

In [ ]:
# Utilização de Dataset Local (arquivo bruto)
df_02 = pd.read_csv('../../dataset/raw/scam_1.csv', encoding='utf-8')  

# Visualização prévia dele
df_02.head()

In [ ]:
# Utilização de Dataset Local (arquivo bruto)
df_03 = pd.read_csv('../../dataset/raw/scam_2.csv', encoding='utf-8')  

# Visualização prévia dele
df_03.head()

In [ ]:
# Utilização de Dataset Local (arquivo bruto)
df_04 = pd.read_csv('../../dataset/raw/scam_3.csv', encoding='utf-8')  

# Visualização prévia dele
df_04.head()

In [ ]:
# Utilização de Dataset Local (arquivo bruto)
df_05 = pd.read_csv('../../dataset/raw/scam_4.csv', encoding='utf-8')  

# Visualização prévia dele
df_05.head()

In [ ]:
# Utilização de Dataset Local (arquivo bruto)
df_06 = pd.read_csv('../../dataset/raw/scam_5.csv', encoding='utf-8')  

# Visualização prévia dele
df_06.head()

In [ ]:
# Utilização de Dataset Local (arquivo bruto)
df_07 = pd.read_csv('../../dataset/raw/emotional_social_engineering_attacks.csv', encoding='utf-8')  

# Visualização prévia dele
df_07.head()

#### Pré-Processamento do .CSV "Bruto"

In [ ]:
import pandas as pd
import spacy

# Carrega o modelo spaCy
nlp = spacy.load("en_core_web_sm")

# Função para limpar e processar o texto dos e-mails usando spaCy:
# - converte para minúsculas
# - remove stopwords, pontuações e tokens não alfabéticos
# - aplica lematização
def preprocess_spacy(text):
    if pd.isnull(text): 
        return ""
    doc = nlp(text.lower())
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct and token.is_alpha]
    return " ".join(tokens)

In [ ]:
# -----------------------------
# Dataset principal (df)
# -----------------------------
df = pd.read_csv('../../dataset/raw/phishing_emails.csv', encoding='utf-8')

# Remove coluna extra se existir
if "Unnamed: 0" in df.columns:
    df.drop(columns=["Unnamed: 0"], inplace=True)

# Padroniza labels
df["Engenharia Social?"] = df["Email Type"].replace({
    "Safe Email": 0,
    "Phishing Email": 1
})

# Renomeia e mantém só as colunas necessárias
df.rename(columns={"Email Text": "Conteudo"}, inplace=True)
df = df[["Conteudo", "Engenharia Social?"]]

In [ ]:
# -----------------------------
# Datasets df02 até df06 (só têm coluna "0")
# Todos são ataques → label 1
# -----------------------------
datasets_scam = []
for i in range(1, 6):  # scam_1.csv até scam_5.csv
    temp = pd.read_csv(f'../../dataset/raw/scam_{i}.csv', encoding='utf-8')
    if "Unnamed: 0" in temp.columns:
        temp.drop(columns=["Unnamed: 0"], inplace=True)
    temp.rename(columns={temp.columns[0]: "Conteudo"}, inplace=True)
    temp["Engenharia Social?"] = 1  # todos são ataques
    datasets_scam.append(temp)

In [ ]:
# -----------------------------
# Dataset df07 (emotional attacks)
# Coluna "Chat Log" → Conteudo
# Coluna "Result" é sempre "Attack" → 1
# -----------------------------
df_07 = pd.read_csv('../../dataset/raw/emotional_social_engineering_attacks.csv', encoding='utf-8')

df_07.rename(columns={"Chat Log": "Conteudo"}, inplace=True)
df_07["Engenharia Social?"] = df_07["Result"].replace({
    "Attack": 1,
    "No Attack": 0
})
df_07 = df_07[["Conteudo", "Engenharia Social?"]]

In [ ]:
# -----------------------------
# Junta todos os datasets
# -----------------------------
final_df = pd.concat([df] + datasets_scam + [df_07], ignore_index=True)

In [ ]:
# -----------------------------
# Pré-processa os textos
# -----------------------------
final_df["Conteudo"] = final_df["Conteudo"].apply(preprocess_spacy)

In [ ]:
# Salva no CSV final
final_df.to_csv("../../dataset/processed/emails_unificado.csv", index=False, encoding="utf-8")

In [ ]:
print(final_df.head())
print(final_df["Engenharia Social?"].value_counts())